# Classificação: tipos de feijão (Dry Bean)

Queremos prever o tipo do feijão (sete classes) a partir de 16 medidas de tamanho e forma do grão. A análise exploratória está no notebook 03. Aqui comparamos os classificadores da `rbg`:

1. **Regressão logística**, com cinco variáveis escolhidas pela correlação.
2. **LDA e QDA**, sem PCA e com PCA, escolhidas por validação cruzada.
3. **KNN** sobre as componentes principais (PCA).
4. **Árvores**: árvore com e sem poda, bagging e random forest.

Todos usam a mesma divisão estratificada (`split_dry_bean`: 10831 grãos de treino e 2712 de teste, sem as linhas duplicadas), e no fim são comparados no mesmo teste. Como as classes são desbalanceadas, além da acurácia reportamos a acurácia balanceada (média do recall das classes) e o F1 macro.

In [ ]:
import time

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

from rbg.Classification import KNN, LDA, QDA, Logistic
from rbg.DecisionTree import ClassificationTree, Bagging, RandomForest
from rbg.ModelSelection import GridSearch, KFold, StratifiedKFold
from rbg.Preprocessing import Standard
from rbg.Unsupervised import PCA
from rbg.Utilities import Accuracy, F1Score, Precision, Recall, error_rate, train_test_split

from utils.dados import (SEED, ALVO_BEAN, FEATURES_BEAN, carregar_dry_bean,
                         split_dry_bean, salvar_resultados)
from utils.avaliacao import metricas_classificacao
from utils.graficos import CLASS_COLORS, configurar_altair, stylize

configurar_altair()

## 1. Preparar os dados

Usamos a [base Dry Bean da UCI](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset), que tem sete tipos de feijão. As linhas duplicadas são removidas antes de separar treino e teste.

In [ ]:
TARGET_CLASS = ALVO_BEAN
FEATURES_CLASS = FEATURES_BEAN

dados_class = carregar_dry_bean()

display(dados_class.head())
display(dados_class[TARGET_CLASS].value_counts())

## 2. Separar treino e teste

Reservamos 20% de cada classe para o teste e usamos os 80% restantes para o treino.

In [ ]:
X_treino_completo, X_teste_completo, y_treino, y_teste = split_dry_bean(dados_class)
classes = np.unique(y_treino)

print('Treino:', X_treino_completo.shape)
print('Teste reservado:', X_teste_completo.shape)

## Parte 1: regressão logística

### 3. Escolha das variáveis

Como várias características medem coisas parecidas, vou manter cinco variáveis. Vou seguir a ordem das colunas e escolher uma variável se sua correlação absoluta com todas as já escolhidas for menor que 0,8, usando apenas o treino.

Com esse critério, vou usar `Area`, `AspectRatio`, `Extent`, `Solidity` e `Roundness`.

In [ ]:
dados_treino = pd.DataFrame(X_treino_completo, columns=FEATURES_CLASS)
correlacoes = dados_treino.corr()
colunas_escolhidas = []

for coluna in FEATURES_CLASS:
    if (correlacoes.loc[coluna, colunas_escolhidas].abs() < 0.8).all():
        colunas_escolhidas.append(coluna)
    if len(colunas_escolhidas) == 5:
        break

print(colunas_escolhidas)
display(correlacoes.loc[colunas_escolhidas, colunas_escolhidas].round(4))

indices_escolhidos = [FEATURES_CLASS.index(coluna) for coluna in colunas_escolhidas]
X_treino = X_treino_completo[:, indices_escolhidos]
X_teste = X_teste_completo[:, indices_escolhidos]

### 4. Ajustar o modelo

As variáveis têm escalas diferentes, então vou padronizá-las usando a média e o desvio do treino. Vou aplicar a mesma transformação no teste.

Vou usar o modelo com intercepto e manter `DERMASON` como referência, por ser a classe mais comum no treino.

In [ ]:
padronizador = Standard()
padronizador.fit(X_treino)
X_treino_padronizado = padronizador.transform(X_treino)
X_teste_padronizado = padronizador.transform(X_teste)

referencia = 'DERMASON'
modelo_final = Logistic(
    classes=classes, reference_class=referencia, fit_intercept=True
)
modelo_final.fit(X_treino_padronizado, y_treino)
print('Referência:', referencia)

### 5. Avaliação no teste

Vou avaliar o modelo com acurácia e acurácia balanceada. Como as classes têm quantidades diferentes de observações, a balanceada permite dar o mesmo peso ao recall de cada uma. Também vou comparar precisão e recall por classe.

In [ ]:
previsoes_teste = modelo_final.predict(X_teste_padronizado)
recall_teste = Recall(previsoes_teste, y_teste)

print(f'Acurácia: {Accuracy(previsoes_teste, y_teste):.4f}')
print(f'Acurácia balanceada: {recall_teste.mean():.4f}')

metricas_classes_teste = pd.DataFrame({
    'classe': classes,
    'precisao': Precision(previsoes_teste, y_teste),
    'recall': recall_teste
})
display(metricas_classes_teste.round(4))

resultado_logistica = metricas_classificacao('Logística (5 variáveis)', previsoes_teste, y_teste)

### 6. Inferência

Vou usar `k_coef()` para mostrar os betas de cada classe em relação a `DERMASON`. Cada cor representa uma variável e a altura da barra mostra seu beta estimado.

Vou deixar o intercepto fora do gráfico e não mostrar `BOMBAY`, por ter menos amostras. Ela continua no modelo e nas métricas.

In [ ]:
classes_inferencia = [classe for classe in classes if classe not in [referencia, 'BOMBAY']]

tabelas = []
for classe in classes_inferencia:
    tabela = pd.DataFrame({
        'classe': classe,
        'variavel': colunas_escolhidas,
        'beta': modelo_final.k_coef(classe)[1:]
    })
    tabelas.append(tabela)

tabela_inferencia = pd.concat(tabelas, ignore_index=True)

alt.Chart(tabela_inferencia).mark_bar().encode(
    x=alt.X('classe:N', title='Tipo de feijão', axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset('variavel:N', sort=colunas_escolhidas),
    y=alt.Y('beta:Q', title='Beta — referência: DERMASON'),
    color=alt.Color('variavel:N', title='Variável', sort=colunas_escolhidas),
    tooltip=['classe:N', 'variavel:N', alt.Tooltip('beta:Q', format='.4f')]
).properties(width=700, height=350)

Com essas barras, conseguimos notar como cada variável está associada à chance de cada tipo de feijão em relação a `DERMASON`, mantendo as outras variáveis fixas. Por exemplo, um `AspectRatio` maior favorece `HOROZ`, enquanto um valor menor favorece `SEKER`, nessa comparação.

A `Area` tem beta positivo em todas as classes mostradas: grãos maiores favorecem essas classes em relação a `DERMASON`. Já `Roundness` tem beta negativo para `BARBUNYA` e positivo para `CALI`, indicando associações em sentidos diferentes.

As barras representam mudanças no logaritmo da razão entre as probabilidades da classe e de `DERMASON`, não aumentos diretos de probabilidade.

### 7. Comentários sobre os resultados

A logística acertou 91,74% do teste. `BOMBAY`, a classe mais distinta, é acertada praticamente sempre; `SIRA` e `BARBUNYA` têm o menor recall, porque se confundem com `DERMASON` e `CALI`.

A escolha das variáveis por correlação reduz a redundância, mas não garante que essas cinco sejam as melhores.

## Parte 2: LDA e QDA

A LDA e a QDA modelam cada classe como uma normal multivariada e classificam pela maior probabilidade a posteriori. A LDA usa uma matriz de covariância agrupada (*pooled*) para todas as classes, o que dá fronteiras lineares; a QDA estima uma matriz por classe, o que dá fronteiras quadráticas e mais parâmetros.

Comparamos oito configurações: LDA e QDA sem PCA e com PCA retendo 90%, 95% e 99% da variância. A escolha é feita por validação cruzada estratificada no treino, e só a configuração escolhida é avaliada no teste.

In [ ]:
# Padronização das 16 variáveis, ajustada no treino (também usada pelo KNN na Parte 3)
padronizador_16 = Standard()
padronizador_16.fit(X_treino_completo)
Z_treino = padronizador_16.transform(X_treino_completo)
Z_teste = padronizador_16.transform(X_teste_completo)

PCA_VARIANCE_TARGET = 0.90
CLASSES_LDA_QDA = classes

### Métricas

As funções da RBG recebem os argumentos na ordem (y_pred, y_true). Accuracy retorna um escalar; Precision, Recall e F1Score retornam vetores por classe, ordenados pelas classes presentes na união de y_pred e y_true. Calculamos as médias macro explicitamente. O F1 ponderado usa o suporte de cada classe. Como a base é desbalanceada, o F1-macro será a métrica principal de comparação.

In [ ]:
def preparar_rotulos(y_pred: np.ndarray, y_true: np.ndarray):
    """Padroniza e valida os vetores de rótulos."""

    y_pred = np.asarray(y_pred).ravel()
    y_true = np.asarray(y_true).ravel()

    if y_pred.shape != y_true.shape:
        raise ValueError("y_pred e y_true precisam ter o mesmo tamanho.")

    if y_true.size == 0:
        raise ValueError("Os vetores de rótulos não podem estar vazios.")

    classes_metricas = np.unique(np.concatenate([y_true, y_pred]))

    return y_pred, y_true, classes_metricas


def metricas_detalhadas(
    y_pred: np.ndarray,
    y_true: np.ndarray
) -> dict[str, float]:
    """Calcula métricas globais utilizando as funções da RBG."""

    y_pred, y_true, classes_metricas = preparar_rotulos(
        y_pred, y_true
    )

    precisao_por_classe = Precision(y_pred, y_true)
    recall_por_classe = Recall(y_pred, y_true)
    f1_por_classe = F1Score(y_pred, y_true)

    # Confere se os vetores retornados seguem a ordem esperada das classes.
    if not all(
        metricas.shape == classes_metricas.shape
        for metricas in (
            precisao_por_classe,
            recall_por_classe,
            f1_por_classe
        )
    ):
        raise ValueError("Dimensões inesperadas nas métricas retornadas pela RBG.")

    # Suporte: número de observações reais pertencentes a cada classe.
    suporte = np.array([
        np.sum(y_true == classe)
        for classe in classes_metricas
    ], dtype=float)

    return {
        "Acurácia": float(Accuracy(y_pred, y_true)),
        "Precisão-macro": float(np.mean(precisao_por_classe)),
        "Recall-macro": float(np.mean(recall_por_classe)),
        "F1-macro": float(np.mean(f1_por_classe)),
        "F1-ponderado": float(
            np.average(f1_por_classe, weights=suporte)
        )
    }


def tabela_metricas_classe(
    y_pred: np.ndarray,
    y_true: np.ndarray
) -> pd.DataFrame:
    """Retorna precisão, recall, F1 e suporte para cada classe."""

    y_pred, y_true, classes_metricas = preparar_rotulos(
        y_pred, y_true
    )

    precisao = Precision(y_pred, y_true)
    recall = Recall(y_pred, y_true)
    f1 = F1Score(y_pred, y_true)

    return pd.DataFrame({
        "Precisão": precisao,
        "Recall": recall,
        "F1-score": f1,
        "Suporte": [
            np.sum(y_true == classe)
            for classe in classes_metricas
        ]
    }, index=pd.Index(classes_metricas, name="Classe"))


def matriz_confusao_rbg(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray
) -> pd.DataFrame:
    """
    Monta a matriz de confusão.
    Linhas representam as classes reais e colunas, as previstas.
    Atenção: aqui a ordem dos argumentos é (y_true, y_pred).
    """

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    classes = np.asarray(classes)

    if y_true.shape != y_pred.shape:
        raise ValueError("y_true e y_pred precisam ter o mesmo tamanho.")

    if y_true.size == 0:
        raise ValueError("Os vetores de rótulos não podem estar vazios.")

    if classes.ndim != 1 or classes.size == 0:
        raise ValueError("A lista de classes precisa ser um vetor não vazio.")

    if np.unique(classes).size != classes.size:
        raise ValueError("A lista de classes não pode conter duplicatas.")

    indice = {classe: i for i, classe in enumerate(classes)}
    matriz = np.zeros((len(classes), len(classes)), dtype=int)

    for verdadeiro, previsto in zip(y_true, y_pred):

        if verdadeiro not in indice or previsto not in indice:
            raise ValueError(
                "Foi encontrada uma classe fora da lista informada."
            )

        matriz[indice[verdadeiro], indice[previsto]] += 1

    return pd.DataFrame(
        matriz,
        index=pd.Index(classes, name="Classe real"),
        columns=pd.Index(classes, name="Classe prevista")
    )

### Diagnóstico das matrizes de covariância

Antes da realização dos experimentos de classificação, investigamos as propriedades numéricas das matrizes de covariância utilizadas pelos classificadores LDA e QDA da biblioteca RBG.

A análise considera dois espaços de representação: o espaço original, composto pelas 16 variáveis padronizadas, e o espaço reduzido pelo PCA, utilizando o limiar de variância explicada de 90%.

Essa investigação é importante porque o LDA utiliza uma matriz de covariância agrupada (*pooled covariance*), enquanto o QDA estima uma matriz de covariância específica para cada classe. A inversão dessas matrizes é uma operação relevante para o funcionamento dos classificadores, de modo que sua estrutura e seu condicionamento numérico merecem atenção.

No espaço original padronizado, todas as matrizes apresentaram posto completo, igual à sua dimensão. Apesar disso, os números de condição foram elevados: aproximadamente 8,82 × 10⁵ para a matriz agrupada do LDA e valores entre 1,06 × 10⁷ e 1,44 × 10⁸ para as matrizes específicas do QDA. Esses valores indicam que, embora as matrizes sejam inversíveis no diagnóstico realizado, algumas apresentam condicionamento numérico desfavorável.

Após a aplicação do PCA, a representação passou a utilizar quatro componentes, responsáveis por reter aproximadamente 94,98% da variância. Nesse espaço, todas as matrizes continuaram apresentando posto completo, enquanto seus números de condição passaram a variar aproximadamente entre 2,59 e 11,48.

Portanto, os resultados indicam que a redução dimensional proporcionada pelo PCA melhora significativamente o condicionamento numérico das matrizes de covariância. Esse comportamento justifica investigar seu efeito nos classificadores, embora uma melhora na estabilidade numérica não implique, necessariamente, maior desempenho preditivo.

Ressaltamos que este diagnóstico possui caráter exploratório e não substitui o ajuste realizado dentro de cada fold da validação cruzada. Nos experimentos, o pré-processamento é novamente ajustado apenas com os dados de treinamento de cada partição.

In [ ]:
def diagnosticar_covariancias(
    Z: np.ndarray, y: np.ndarray, descricao: str
) -> list[dict[str, object]]:
    classes = np.unique(y)
    n, p = Z.shape

    residuos = np.vstack([
        Z[y == classe] - Z[y == classe].mean(axis=0)
        for classe in classes
    ])
    cov_lda = (residuos.T @ residuos) / (n - len(classes))

    matrizes = [("LDA — pooled", cov_lda)]
    for classe in classes:
        Z_classe = Z[y == classe]
        centrado = Z_classe - Z_classe.mean(axis=0)
        cov_qda = (centrado.T @ centrado) / (len(Z_classe) - 1)
        matrizes.append((f"QDA — {classe}", cov_qda))

    linhas = []
    for nome, cov in matrizes:
        linhas.append({
            "Espaço": descricao,
            "Matriz": nome,
            "Dimensão": p,
            "Posto": int(np.linalg.matrix_rank(cov)),
            "Número de condição": float(np.linalg.cond(cov)),
        })
    return linhas


scaler_diag = Standard().fit(X_treino_completo)
X_train_diag_std = scaler_diag.transform(X_treino_completo)
pca_diag = PCA(n_components=PCA_VARIANCE_TARGET).fit(X_train_diag_std)
X_train_diag_pca = pca_diag.transform(X_train_diag_std)

linhas_diagnostico = (
    diagnosticar_covariancias(
        X_train_diag_std, y_treino, "16 features padronizadas"
    )
    + diagnosticar_covariancias(
        X_train_diag_pca, y_treino, "PCA 90%"
    )
)

df_diagnostico_cov = pd.DataFrame(linhas_diagnostico)
display(df_diagnostico_cov.round(3))
print(
    f"PCA reteve {pca_diag.components.shape[0]} componentes e "
    f"{np.sum(pca_diag.explained_variance_ratio) * 100:.2f}% da variância."
)

### Validação cruzada estratificada

A comparação dos classificadores é realizada por meio de validação cruzada estratificada com cinco folds, utilizando exclusivamente o conjunto de treinamento externo. Todas as configurações utilizam exatamente as mesmas partições, permitindo uma comparação consistente entre os resultados.

São investigadas oito configurações: LDA e QDA sem redução dimensional e os mesmos classificadores com PCA, considerando limiares de variância explicada de 90%, 95% e 99%.

Em cada fold, o `Standard` é ajustado apenas com as observações pertencentes ao subconjunto de treinamento. Quando há redução dimensional, o PCA também é ajustado exclusivamente nesse subconjunto. Em seguida, as transformações são aplicadas às observações de validação, preservando a separação necessária para uma avaliação adequada.

A principal métrica de comparação é o F1-macro médio, que atribui o mesmo peso ao desempenho de cada classe, independentemente de sua frequência. Também são consideradas a acurácia, a precisão-macro, o recall-macro e o F1 ponderado. O desvio-padrão do F1-macro entre os folds é utilizado como medida descritiva da variação do desempenho entre as partições.

É importante observar que os percentuais associados ao PCA representam limiares mínimos de variância explicada. Como o número de componentes é inteiro, a variância efetivamente retida pode ultrapassar o valor solicitado.

In [ ]:
N_FOLDS_LDA_QDA = 5

cv_lda_qda = StratifiedKFold(
    k=N_FOLDS_LDA_QDA,
    shuffle=True,
    random_state=SEED
)

# Guardamos os folds para garantir que todos os experimentos utilizem
# exatamente as mesmas divisões de treino e validação.
folds_cv = list(cv_lda_qda.split(X_treino_completo, y_treino))


def executar_experimento_cv(
    nome_modelo: str,
    classe_modelo,
    usar_pca: bool
) -> pd.DataFrame:

    resultados = []

    for numero_fold, (idx_tr, idx_val) in enumerate(folds_cv, start=1):

        X_fold_tr = X_treino_completo[idx_tr]
        X_fold_val = X_treino_completo[idx_val]

        y_fold_tr = y_treino[idx_tr]
        y_fold_val = y_treino[idx_val]

        # 1. Padronização: ajustada somente no fold de treino.
        scaler_fold = Standard().fit(X_fold_tr)

        X_fold_tr_std = scaler_fold.transform(X_fold_tr)
        X_fold_val_std = scaler_fold.transform(X_fold_val)

        # 2. PCA: aplicado somente quando o experimento exigir.
        if usar_pca:

            pca_fold = PCA(
                n_components=PCA_VARIANCE_TARGET
            ).fit(X_fold_tr_std)

            X_fold_tr_modelo = pca_fold.transform(X_fold_tr_std)
            X_fold_val_modelo = pca_fold.transform(X_fold_val_std)

            numero_componentes = pca_fold.components.shape[0]

            variancia_retida = float(
                np.sum(pca_fold.explained_variance_ratio)
            )

        else:

            X_fold_tr_modelo = X_fold_tr_std
            X_fold_val_modelo = X_fold_val_std

            numero_componentes = X_fold_tr_std.shape[1]
            variancia_retida = 1.0

        # 3. Ajuste e previsão do classificador da RBG.
        erro_ajuste = None

        try:

            modelo_fold = classe_modelo(
                labels=CLASSES_LDA_QDA
            )

            modelo_fold.fit(X_fold_tr_modelo, y_fold_tr)

            y_fold_pred = modelo_fold.predict(X_fold_val_modelo)

            metricas_fold = metricas_detalhadas(
                y_fold_pred,
                y_fold_val
            )

        except (ValueError, np.linalg.LinAlgError) as erro:

            # Registramos eventuais problemas numéricos sem interromper
            # a execução dos outros folds e experimentos.
            metricas_fold = {
                "Acurácia": np.nan,
                "Precisão-macro": np.nan,
                "Recall-macro": np.nan,
                "F1-macro": np.nan,
                "F1-ponderado": np.nan,
            }

            erro_ajuste = str(erro)

        resultados.append({
            "Modelo": nome_modelo,
            "Fold": numero_fold,
            "Componentes": numero_componentes,
            "Variância retida": variancia_retida,
            **metricas_fold,
            "Erro de ajuste": erro_ajuste,
        })

    return pd.DataFrame(resultados)

#### LDA sem PCA

In [ ]:
resultados_lda_sem_pca = executar_experimento_cv(
    nome_modelo="LDA sem PCA",
    classe_modelo=LDA,
    usar_pca=False
)

print("Resultados do LDA sem PCA:")
display(resultados_lda_sem_pca.round(4))

#### QDA sem PCA

In [ ]:
resultados_qda_sem_pca = executar_experimento_cv(
    nome_modelo="QDA sem PCA",
    classe_modelo=QDA,
    usar_pca=False
)

print("Resultados do QDA sem PCA:")
display(resultados_qda_sem_pca.round(4))

#### LDA com PCA

In [ ]:
resultados_lda_com_pca = executar_experimento_cv(
    nome_modelo="LDA com PCA",
    classe_modelo=LDA,
    usar_pca=True
)

print("Resultados do LDA com PCA:")
display(resultados_lda_com_pca.round(4))

#### QDA com PCA

In [ ]:
resultados_qda_com_pca = executar_experimento_cv(
    nome_modelo="QDA com PCA",
    classe_modelo=QDA,
    usar_pca=True
)

print("Resultados do QDA com PCA:")
display(resultados_qda_com_pca.round(4))

Sem PCA, a QDA teve F1 macro médio de 0,9229 na validação cruzada, contra 0,9185 da LDA. Isso sugere que estimar uma covariância para cada classe representa melhor as diferenças entre as variedades de feijão. Com PCA de 90%, as duas pioraram (0,8910 e 0,8826).

### Análise de sensibilidade do limiar de PCA

Após os experimentos principais, investigamos como a escolha do limiar de variância explicada pelo PCA influencia o desempenho dos classificadores.

Como as configurações com limiar de 90% já foram avaliadas na etapa anterior, esta análise adicional considera os limiares de 95% e 99%, tanto para LDA quanto para QDA.

Para cada configuração, utilizamos as mesmas cinco partições da validação cruzada. Em cada fold, o `Standard` e o PCA são ajustados exclusivamente com as observações de treinamento, sendo posteriormente aplicados às observações de validação.

Dessa maneira, os resultados permitem comparar os diferentes níveis de redução dimensional sem utilizar o conjunto de teste externo e sem introduzir informações das observações de validação durante o ajuste do pré-processamento.

In [ ]:
def avaliar_sensibilidade_pca(
    limiares: tuple[float, ...] = (0.95, 0.99)
) -> tuple[pd.DataFrame, pd.DataFrame]:

    resultados = []

    for limiar in limiares:

        for nome_modelo, classe_modelo in [
            ("LDA", LDA),
            ("QDA", QDA)
        ]:

            for fold, (idx_tr, idx_val) in enumerate(folds_cv, start=1):

                Xtr = X_treino_completo[idx_tr]
                Xval = X_treino_completo[idx_val]

                ytr = y_treino[idx_tr]
                yval = y_treino[idx_val]

                scaler = Standard().fit(Xtr)

                Xtr_std = scaler.transform(Xtr)
                Xval_std = scaler.transform(Xval)

                pca = PCA(
                    n_components=float(limiar)
                ).fit(Xtr_std)

                Xtr_pca = pca.transform(Xtr_std)
                Xval_pca = pca.transform(Xval_std)

                erro_ajuste = None

                try:
                    modelo = classe_modelo(
                        labels=CLASSES_LDA_QDA
                    )

                    modelo.fit(Xtr_pca, ytr)

                    pred = modelo.predict(Xval_pca)

                    metricas_fold = metricas_detalhadas(
                        pred,
                        yval
                    )

                except (ValueError, np.linalg.LinAlgError) as erro:

                    metricas_fold = {
                        "Acurácia": np.nan,
                        "Precisão-macro": np.nan,
                        "Recall-macro": np.nan,
                        "F1-macro": np.nan,
                        "F1-ponderado": np.nan
                    }

                    erro_ajuste = str(erro)

                resultados.append({
                    "Limiar PCA": limiar,
                    "Modelo": nome_modelo,
                    "Fold": fold,
                    **metricas_fold,
                    "Componentes": pca.components.shape[0],
                    "Variância retida": float(
                        np.sum(pca.explained_variance_ratio)
                    ),
                    "Erro de ajuste": erro_ajuste
                })

    detalhe = pd.DataFrame(resultados)

    resumo = detalhe.groupby(
        ["Limiar PCA", "Modelo"]
    ).agg(
        F1_macro_medio=("F1-macro", "mean"),
        F1_macro_dp=("F1-macro", "std"),
        Acuracia_media=("Acurácia", "mean"),
        Precisao_macro_media=("Precisão-macro", "mean"),
        Recall_macro_medio=("Recall-macro", "mean"),
        F1_ponderado_medio=("F1-ponderado", "mean"),
        Folds_validos=("F1-macro", "count"),
        Componentes_medio=("Componentes", "mean"),
        Variancia_retida_media=("Variância retida", "mean")
    ).reset_index()

    return detalhe, resumo


df_sensibilidade_detalhe, df_sensibilidade_resumo = (
    avaliar_sensibilidade_pca()
)

print("Análise de sensibilidade do PCA:")
display(df_sensibilidade_resumo.round(4))

Reduzir para 90% ou 95% da variância prejudicou a classificação, mas reter 99% (7 componentes, em média 99,6% da variância) melhorou a QDA: ela chegou a um F1 macro de 0,9306, acima da QDA sem PCA (0,9229), e com o menor desvio entre os folds (0,0060). Para a LDA, nenhum limiar superou o modelo sem PCA. O efeito do PCA, portanto, depende do limiar e do classificador.

In [ ]:
# Consolidação dos oito cenários avaliados na validação cruzada.

def resumir_configuracao(
    nome: str,
    dados: pd.DataFrame
) -> dict:

    return {
        "Configuração": nome,
        "F1_macro_medio": dados["F1-macro"].mean(),
        "F1_macro_dp": dados["F1-macro"].std(),
        "Acuracia_media": dados["Acurácia"].mean(),
        "Precisao_macro_media": dados["Precisão-macro"].mean(),
        "Recall_macro_medio": dados["Recall-macro"].mean(),
        "F1_ponderado_medio": dados["F1-ponderado"].mean(),
        "Folds_validos": dados["F1-macro"].count(),
        "Componentes_medio": dados["Componentes"].mean(),
        "Variancia_retida_media": dados["Variância retida"].mean()
    }


linhas_resumo = []

# 1. Modelos sem PCA e modelos com PCA de 90%.
resultados_principais = [
    ("LDA sem PCA", resultados_lda_sem_pca),
    ("QDA sem PCA", resultados_qda_sem_pca),
    ("LDA — PCA 90%", resultados_lda_com_pca),
    ("QDA — PCA 90%", resultados_qda_com_pca)
]

for nome, dados in resultados_principais:
    linhas_resumo.append(
        resumir_configuracao(nome, dados)
    )


# Modelos com PCA de 95% e 99%.
for (limiar, nome_modelo), grupo in (
    df_sensibilidade_detalhe.groupby(
        ["Limiar PCA", "Modelo"]
    )
):

    nome_configuracao = (
        f"{nome_modelo} — PCA {limiar:.0%}"
    )

    linhas_resumo.append(
        resumir_configuracao(nome_configuracao, grupo)
    )


# Tabela consolidada.
resumo_cv_completo = (
    pd.DataFrame(linhas_resumo)
    .set_index("Configuração")
)

# Uma configuração só é elegível se tiver os cinco folds válidos.
resumo_cv_completo["Elegível"] = (
    resumo_cv_completo["Folds_validos"] == N_FOLDS_LDA_QDA
)

print("Comparação completa das oito configurações:")
display(
    resumo_cv_completo.sort_values(
        "F1_macro_medio",
        ascending=False
    ).round(4)
)


# Seleção do melhor modelo sem consultar o teste externo.
resumo_cv_validos = resumo_cv_completo[
    resumo_cv_completo["Elegível"]
]

if resumo_cv_validos.empty:
    raise RuntimeError(
        "Nenhuma configuração completou os cinco folds."
    )

# Maior F1-macro médio; em caso de empate, menor desvio-padrão.
ranking_cv = resumo_cv_validos.sort_values(
    by=["F1_macro_medio", "F1_macro_dp"],
    ascending=[False, True]
)

melhor_configuracao_cv = ranking_cv.index[0]

print("\nConfiguração selecionada pela validação cruzada:")
print(melhor_configuracao_cv)

display(
    ranking_cv.head(1).round(4)
)

## Ajuste final e avaliação no conjunto de teste

A configuração com maior F1 macro médio na validação, usando o desvio entre os folds como desempate, foi a QDA com PCA de 99%. Reajustamos o `Standard` e o PCA com todo o treino, aplicamos as transformações ao teste, que ficou separado durante toda a seleção, e ajustamos a QDA no treino inteiro.

Também incluímos um baseline que prevê sempre a classe mais frequente do treino, `DERMASON`, para verificar se o modelo discrimina melhor do que uma regra que só olha a frequência das classes.

In [ ]:
# Associação entre os nomes da validação cruzada e os classificadores.
configuracoes = {
    "LDA sem PCA": (LDA, None),
    "QDA sem PCA": (QDA, None),
    "LDA — PCA 90%": (LDA, 0.90),
    "QDA — PCA 90%": (QDA, 0.90),
    "LDA — PCA 95%": (LDA, 0.95),
    "QDA — PCA 95%": (QDA, 0.95),
    "LDA — PCA 99%": (LDA, 0.99),
    "QDA — PCA 99%": (QDA, 0.99)
}

if melhor_configuracao_cv not in configuracoes:
    raise ValueError("Configuração selecionada não reconhecida.")

classe_modelo_final_lda_qda, limiar_pca_final = configuracoes[
    melhor_configuracao_cv
]


# 1. Ajuste da padronização somente no treino externo.
scaler_final = Standard().fit(X_treino_completo)

X_train_final_std = scaler_final.transform(X_treino_completo)
X_test_final_std = scaler_final.transform(X_teste_completo)


# 2. Aplicação do PCA somente se a configuração selecionada exigir.
pca_final = None

if limiar_pca_final is not None:

    pca_final = PCA(
        n_components=limiar_pca_final
    ).fit(X_train_final_std)

    X_train_final = pca_final.transform(X_train_final_std)
    X_test_final = pca_final.transform(X_test_final_std)

    print(
        f"Variância retida: "
        f"{np.sum(pca_final.explained_variance_ratio) * 100:.2f}%"
    )

else:

    X_train_final = X_train_final_std
    X_test_final = X_test_final_std

    print("Modelo ajustado sem PCA.")


# 3. Ajuste do classificador vencedor e previsão no teste externo.
modelo_final_lda_qda = classe_modelo_final_lda_qda(
    labels=CLASSES_LDA_QDA
)

modelo_final_lda_qda.fit(X_train_final, y_treino)

y_pred_final = modelo_final_lda_qda.predict(X_test_final)


# 4. Avaliação externa da configuração selecionada.
predicoes_lda_qda = {
    melhor_configuracao_cv: y_pred_final
}

resultados_teste_lda_qda = [{
    "Modelo": melhor_configuracao_cv,
    **metricas_detalhadas(y_pred_final, y_teste)
}]


# 5. Baseline de classe majoritária.
classe_majoritaria_treino = (
    pd.Series(y_treino).value_counts().idxmax()
)

y_pred_baseline = np.full(
    y_teste.shape,
    classe_majoritaria_treino,
    dtype=y_treino.dtype
)

predicoes_lda_qda["Baseline majoritária"] = y_pred_baseline

resultados_teste_lda_qda.append({
    "Modelo": "Baseline majoritária",
    **metricas_detalhadas(y_pred_baseline, y_teste)
})


# 6. Resultados finais.
df_teste_lda_qda = (
    pd.DataFrame(resultados_teste_lda_qda)
    .set_index("Modelo")
)

print("Configuração selecionada:", melhor_configuracao_cv)
print("Classe majoritária do treino:", classe_majoritaria_treino)

print("\nAvaliação no conjunto de teste externo:")
display(df_teste_lda_qda.round(4))

### Diagnóstico por classe e matrizes de confusão

Além das métricas globais, olhamos o desempenho da QDA com PCA de 99% em cada uma das sete variedades: precisão, recall, F1 e suporte, junto com a matriz de confusão (linhas são as classes reais e colunas as previstas). O baseline também aparece, para mostrar como uma regra que sempre prevê `DERMASON` se comporta.

In [ ]:
for nome_modelo, y_pred in predicoes_lda_qda.items():

    print(f"\n===== {nome_modelo}: métricas por classe =====")

    display(
        tabela_metricas_classe(y_pred, y_teste).round(4)
    )

    print(
        f"Matriz de confusão — {nome_modelo} "
        "(linhas=real; colunas=previsto):"
    )

    display(
        matriz_confusao_rbg(
            y_teste,
            y_pred,
            CLASSES_LDA_QDA
        )
    )

`BOMBAY` teve F1 de 1,00: as 105 amostras do teste foram classificadas corretamente. `SEKER` (0,948), `CALI` (0,943) e `HOROZ` (0,936) também tiveram resultados altos. `SIRA` teve o menor F1 (0,860), seguida de `DERMASON` (0,913) e `BARBUNYA` (0,924).

A matriz de confusão mostra de onde vêm esses erros: 64 grãos de `DERMASON` foram classificados como `SIRA` e 32 de `SIRA` como `DERMASON`, a principal confusão do teste. Entre `BARBUNYA` e `CALI`, 22 grãos de `BARBUNYA` foram previstos como `CALI`, e 3 no sentido contrário. São os mesmos pares que a análise exploratória já mostrava sobrepostos, embora a matriz sozinha não explique a causa dessas semelhanças.

A tabela abaixo leva LDA e QDA com as 16 variáveis e a QDA com PCA de 99% para a comparação final do notebook.

In [ ]:
# Entradas para a comparação final do notebook
resultados_discriminantes = []
for nome, classe_modelo in [('LDA', LDA), ('QDA', QDA)]:
    modelo = classe_modelo(labels=classes)
    modelo.fit(Z_treino, y_treino)
    resultados_discriminantes.append(
        metricas_classificacao(f'{nome} (16 variáveis)', modelo.predict(Z_teste), y_teste))
if limiar_pca_final is not None:
    resultados_discriminantes.append(
        metricas_classificacao(melhor_configuracao_cv, y_pred_final, y_teste))
pd.DataFrame(resultados_discriminantes).round(4)

### Análise dos resultados

A QDA foi melhor que a LDA nas quatro condições de dimensionalidade. A melhor configuração na validação cruzada foi a QDA com PCA de 99% (F1 macro de 0,9306 ± 0,0060), seguida da QDA sem PCA (0,9229) e da LDA sem PCA (0,9185).

No teste, a QDA com PCA de 99% teve acurácia de 0,9192 e F1 macro de 0,9321, próximo da média da validação, então o desempenho estimado na validação se manteve no teste. O baseline teve acurácia de 0,2618 e F1 macro de 0,0593, o que mostra que o resultado do modelo não vem só da predominância de `DERMASON`.

No teste, a diferença em relação à QDA com as 16 variáveis é pequena (91,92% contra 91,67% de acurácia). A vantagem do PCA de 99% está mais na estabilidade entre os folds e no condicionamento das matrizes do que em um ganho grande de acerto. Essas conclusões valem para esta divisão, estes dados e as implementações da `rbg`.

## Parte 3: KNN com PCA

### Pré-processamento

O PCA e o KNN dependem da escala das variáveis: sem padronizar, `Area` (dezenas de milhares) dominaria as distâncias e as componentes, e `Solidity` (entre 0 e 1) quase não contaria. A média e o desvio são calculados só no treino e aplicados ao teste, para que o teste não influencie o modelo. Reaproveitamos a padronização das 16 variáveis feita para a LDA e a QDA.

### Redução de dimensionalidade

Várias variáveis medem quase a mesma coisa (`Area`, `Perimeter`, `ConvexArea`, `EquivDiameter`...). O PCA troca essas variáveis correlacionadas por componentes ortogonais, e com menos dimensões as distâncias do KNN ficam mais informativas e mais rápidas de calcular. Mantemos o número de componentes necessário para reter 95% da variância do treino.

In [ ]:
pca = PCA(n_components=0.95)  # variância acumulada desejada
pca.fit(Z_treino)

X_train_pca = pca.transform(Z_treino)
X_test_pca = pca.transform(Z_teste)

print(f"Dimensão original das features: {Z_treino.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Com 4 componentes a variância acumulada fica em 94,98%, logo abaixo do corte, então o PCA mantém 5 componentes (cerca de 97,7% da variância). Mesmo assim, passamos de 16 para 5 dimensões.

### Escolha do k

O $k$ é escolhido por `GridSearch` com `StratifiedKFold` (5 partes, mantendo a proporção das classes), entre valores ímpares para evitar empates na votação, maximizando o F1 macro, que é mais adequado do que a acurácia com classes desbalanceadas.

Como o KNN da `rbg` calcula a distância para todos os pontos de treino a cada previsão, a validação cruzada no treino inteiro seria lenta. Por isso a busca é feita em uma subamostra aleatória de 1000 grãos do treino (com a semente do projeto). Depois o modelo com o melhor $k$ é ajustado no treino completo e avaliado uma única vez no teste.

In [ ]:
_, X_mini, _, y_mini = train_test_split(X_train_pca, y_treino, test_size=1000, random_state=SEED)

def f1_macro(previsto, real):
    return F1Score(previsto, real).mean()

valores_de_k = [3, 5, 7, 9, 11, 15]
grid = GridSearch(
    model=KNN,
    param_grid={'k': valores_de_k},
    metrics=f1_macro,
    greater_is_better=True,
    cv=StratifiedKFold(k=5, shuffle=True, random_state=SEED)
)

grid.fit(X_mini, y_mini)
best_k = grid.best_params_['k']
print("Melhor hiperparâmetro encontrado:", best_k)
print(f"F1 macro médio na validação cruzada: {grid.best_score_ * 100:.2f}%")

### Avaliação final do KNN

In [ ]:
inicio = time.time()
best_knn = KNN(k=best_k)
best_knn.fit(X_train_pca, y_treino)
y_pred_knn = best_knn.predict(X_test_pca)

resultado_knn = metricas_classificacao(f'KNN (k = {best_k}, PCA com {X_train_pca.shape[1]} componentes)', y_pred_knn, y_teste)
print(f"tempo de previsão do teste: {time.time() - inicio:.0f} s")
pd.DataFrame([resultado_knn]).round(4)

### Visualização em duas componentes

Para enxergar por que o KNN erra, projetamos o treino nas duas primeiras componentes principais.

In [ ]:
pca2d = PCA(n_components=2)
pca2d.fit(Z_treino)
X_train_pca2d = pca2d.transform(Z_treino)
print(f"Variância acumulada nas duas componentes: {np.sum(pca2d.explained_variance_ratio) * 100:.2f}%")

df_pca_plot = pd.DataFrame({"PC1": X_train_pca2d[:, 0], "PC2": X_train_pca2d[:, 1], "Class": y_treino})
color_scale = alt.Scale(domain=list(CLASS_COLORS.keys()), range=list(CLASS_COLORS.values()))

stylize(
    alt.Chart(df_pca_plot)
    .mark_circle(size=20, opacity=0.6)
    .encode(
        x=alt.X("PC1:Q", title="Componente Principal 1 (PC1)"),
        y=alt.Y("PC2:Q", title="Componente Principal 2 (PC2)"),
        color=alt.Color("Class:N", scale=color_scale, title="Classe"),
    )
    .properties(title="Dispersão das amostras de treino nas duas primeiras componentes", width=650, height=450)
)

O KNN com k = 5 teve a maior acurácia do notebook, 92,51%. A projeção nas duas primeiras componentes mostra onde ele ainda erra: `BOMBAY` fica isolada, mas `CALI` e `BARBUNYA` se sobrepõem, e `SIRA` fica na fronteira com `DERMASON` e `SEKER`, onde os vizinhos de um grão muitas vezes são de outra classe. As duas componentes são só para visualizar; o modelo usa as cinco.

## Parte 4: árvores

Faremos agora o uso de árvores para prever os nossos dados: árvores de classificação com e sem poda, bagging e random forest. Como no KNN, usamos as componentes do PCA (`X_train_pca` e `X_test_pca`) em vez das 16 colunas originais: com menos colunas, a árvore tem menos variáveis em que procurar o ponto de corte.

* **Validação**: para árvores, random forest e bagging, usaremos o GridSearch para procurar, em cada um dos três, os melhores hiperparâmetros.
* **Comparação de modelos**: com os modelos ajustados pelos seus melhores hiperparâmetros, faremos uma validação cruzada com os dados de treino para decidir qual o melhor modelo.
* **Avaliação final**: dado o modelo escolhido, faremos a predição dos dados de teste e calcularemos seu erro (`error_rate`, que é 1 menos a acurácia).
* **Interpretação dos resultados**: considerações sobre a escolha das métricas e o trade-off entre viés e variância.

### Tamanho da amostra

A construção das árvores na `rbg` testa todos os valores de cada variável como ponto de corte em cada nó, e o tempo cresce rápido com o número de linhas: uma árvore com 4000 grãos já leva perto de um minuto, e o treino tem 10831. Com GridSearch e K-Fold isso seria multiplicado por dezenas de ajustes. Por isso, em toda esta parte as árvores são treinadas em uma subamostra aleatória de 1500 grãos do treino (sorteada com a semente do projeto). O teste continua sendo o mesmo, com os 2712 grãos.

In [ ]:
rng = np.random.default_rng(SEED)
indices_sub = rng.choice(len(y_treino), size=1500, replace=False)
X_sub = X_train_pca[indices_sub]
y_sub = y_treino[indices_sub]

pd.DataFrame({'treino completo': pd.Series(y_treino).value_counts(normalize=True),
              'subamostra': pd.Series(y_sub).value_counts(normalize=True)}).round(3)

In [ ]:
inicio = time.time()
arvore = ClassificationTree()

arvore.fit(X_sub, y_sub)

y_previsto = arvore.predict(X_test_pca)

erro_arvore_sem_poda = error_rate(y_previsto, y_teste)

print("erro da árvore sem poda:", erro_arvore_sem_poda)
print(f"tempo: {time.time() - inicio:.0f} s")

In [ ]:
inicio = time.time()
arvore_poda = ClassificationTree()

arvore_poda.fit(X_sub, y_sub, prune=True)

y_previsto_poda = arvore_poda.predict(X_test_pca)

erro_arvore_com_poda = error_rate(y_previsto_poda, y_teste)

print("erro da árvore com poda:", erro_arvore_com_poda)
print(f"tempo: {time.time() - inicio:.0f} s")

Com o padrão de 100 árvores, o bagging com 5000 linhas levou mais de 60 minutos nos nossos testes. Para ser computacionalmente viável, tanto o random forest quanto o bagging usam apenas 2 ou 3 árvores nesta parte.

In [ ]:
inicio = time.time()
arvore_random = RandomForest(n_trees=3)

arvore_random.fit(X_sub, y_sub)

y_previsto_random = arvore_random.predict(X_test_pca)

erro_arvore_random = error_rate(y_previsto_random, y_teste)

print("erro do random forest:", erro_arvore_random)
print(f"tempo: {time.time() - inicio:.0f} s")

In [ ]:
inicio = time.time()
arvorebag = Bagging(n_trees=3)

arvorebag.fit(X_sub, y_sub)

y_previstobag = arvorebag.predict(X_test_pca)

erro_arvore_bag = error_rate(y_previstobag, y_teste)

print("erro do bagging:", erro_arvore_bag)
print(f"tempo: {time.time() - inicio:.0f} s")

### Validação

Uma vez que árvores (até mesmo as mais simples, sem poda e sem bagging) têm um custo grande para serem montadas de acordo com o algoritmo implementado, e uma vez que as árvores possuem vários parâmetros, então para fazer uma validação otimizada usaremos apenas os hiperparâmetros:

- min_samples_split (e não usaremos min_samples_leaf, que ficará padronizado como 1)
- max_depth (e não usaremos max_leaf)
- min_impurity_decrease
- criterion_split ("Standard" é o índice de Gini; a outra opção é a entropia)

Dentro de criterion_choose, iremos avaliar apenas a moda, já que a média não faz sentido em um problema de classificação, e a mediana também não, porque os tipos de feijão não têm uma ordem. Também deixaremos de fora "min" e "max". A métrica de validação é o `error_rate`.

Todas as validações usam K-Fold com 5 partes, embaralhado com a semente do projeto.

In [ ]:
p_grid = {
    'max_depth': [5],
    'min_samples_split': [10, 50],
    'min_impurity_decrease': [0.1],
    'criterion_split': ['Standard', 'entropy'],
    'criterion_choose': ['Standard']
}

grid_arvore = GridSearch(model=ClassificationTree(), param_grid=p_grid, metrics=error_rate,
                         cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_arvore.fit(X_sub, y_sub)
print(f"GridSearch da árvore: {time.time() - inicio:.0f} s")
print(grid_arvore.best_params_)
print("erro médio na validação:", grid_arvore.best_score_)

Para o bagging, que constrói as árvores uma de cada vez (sem paralelização), testamos menos combinações ainda e variamos o número de árvores entre 2 e 3.

In [ ]:
pbaggin_grid = {
    'max_depth': [7],
    'min_samples_split': [10],
    'min_impurity_decrease': [0.1],
    'criterion_split': ['entropy'],
    'criterion_choose': ['Standard'],
    'n_trees': [3, 2]
}

grid_bagging = GridSearch(model=Bagging(), param_grid=pbaggin_grid, metrics=error_rate,
                          cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_bagging.fit(X_sub, y_sub)
print(f"GridSearch do bagging: {time.time() - inicio:.0f} s")
print(grid_bagging.best_params_)
print("erro médio na validação:", grid_bagging.best_score_)

Similarmente, para o random forest testamos 2 e 3 árvores, com o número de variáveis sorteadas em cada corte igual ao piso da raiz quadrada do total. Como usamos as componentes do PCA, temos 5 variáveis e, portanto, 2 são sorteadas em cada corte.

In [ ]:
p_random_grid = {
    'max_depth': [7],
    'min_samples_split': [10],
    'min_impurity_decrease': [0.1],
    'criterion_split': ['entropy'],
    'criterion_choose': ['Standard'],
    'n_trees': [3, 2],
    'factor_selection': [np.sqrt]
}

grid_Random = GridSearch(model=RandomForest(), param_grid=p_random_grid, metrics=error_rate,
                         cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_Random.fit(X_sub, y_sub)
print(f"GridSearch do random forest: {time.time() - inicio:.0f} s")
print(grid_Random.best_params_)
print("erro médio na validação:", grid_Random.best_score_)

### Seleção de modelo

Para a seleção de modelos, faremos um K-Fold usando os campeões de cada GridSearch: treinamos cada um nas partes de treino e escolhemos aquele que teve o menor `error_rate` médio. Usamos uma semente diferente da do GridSearch para que as partes não sejam as mesmas que escolheram os hiperparâmetros.

In [ ]:
param_r = grid_Random.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

bagging = Bagging(**param_bagging)
arvore_pos_grid = ClassificationTree(**param_arvore)
random_forest = RandomForest(**param_r)

In [ ]:
inicio = time.time()
k_fold = KFold(k=5, shuffle=True, random_state=SEED + 1)
lista_erros_arvore = []
lista_erros_bag = []
lista_erros_random = []

for treino_k, teste_k in k_fold.split(X_sub):
    X_treino_k, X_teste_k = X_sub[treino_k], X_sub[teste_k]
    y_treino_k, y_teste_k = y_sub[treino_k], y_sub[teste_k]

    # árvore nova a cada parte: refazer o fit no mesmo objeto de árvore da rbg
    # reaproveita estado do ajuste anterior e piora a árvore
    arvore_pos_grid = ClassificationTree(**param_arvore)
    random_forest.fit(X_treino_k, y_treino_k)
    bagging.fit(X_treino_k, y_treino_k)
    arvore_pos_grid.fit(X_treino_k, y_treino_k)
    lista_erros_arvore.append(error_rate(arvore_pos_grid.predict(X_teste_k), y_teste_k))
    lista_erros_bag.append(error_rate(bagging.predict(X_teste_k), y_teste_k))
    lista_erros_random.append(error_rate(random_forest.predict(X_teste_k), y_teste_k))

erros_kfold = pd.DataFrame({
    'modelo': ['árvore', 'bagging', 'random forest'],
    'erro médio': [np.mean(lista_erros_arvore), np.mean(lista_erros_bag), np.mean(lista_erros_random)],
})
print(f"tempo: {time.time() - inicio:.0f} s")
erros_kfold.round(4)

### Avaliação final

Com o modelo ganhador, treinamos na subamostra inteira e calculamos o erro no teste. Os outros dois também são ajustados, para entrar na tabela comparativa.

In [ ]:
arvore_pos_grid = ClassificationTree(**param_arvore)  # objeto novo para o ajuste final
modelos_arvore = {'árvore': arvore_pos_grid, 'bagging': bagging, 'random forest': random_forest}
vencedor = erros_kfold.sort_values('erro médio').iloc[0]['modelo']
print("vencedor:", vencedor)

previsoes_arvores = {}
for nome, modelo in modelos_arvore.items():
    modelo.fit(X_sub, y_sub)
    previsoes_arvores[nome] = modelo.predict(X_test_pca)

print("Erro do modelo no teste:", error_rate(previsoes_arvores[vencedor], y_teste))

O bagging guarda quais grãos ficaram fora de cada amostra bootstrap, o que permite calcular o erro OOB (out-of-bag) sem usar o teste. O método `OOB()` da `rbg` converte as previsões para número e por isso só funciona quando as classes são números; aqui as classes são nomes, então fazemos a mesma conta usando os atributos que o bagging guarda (`list_of_trees` e `list_of_indx_test`).

In [ ]:
previsoes_oob, reais_oob = [], []
for i in range(len(y_sub)):
    arvores_fora = [arv for arv, fora in zip(bagging.list_of_trees, bagging.list_of_indx_test) if i in fora]
    if arvores_fora:
        previsoes_oob.append(bagging.predict(X_sub[i:i + 1], trees=arvores_fora)[0])
        reais_oob.append(y_sub[i])

print(f"grãos com previsão OOB: {len(reais_oob)} de {len(y_sub)}")
print("erro OOB do bagging:", error_rate(np.array(previsoes_oob), np.array(reais_oob)))

### Interpretação dos resultados

Na subamostra de 1.500 grãos, a árvore com poda acertou 89,6% do teste, a árvore do GridSearch 89,3% e o bagging com 3 árvores 89,2%. Com tão poucas árvores, o bagging não chegou a ganhar da árvore única. O random forest caiu para 75,5%: com 3 árvores e só 2 das 5 componentes sorteadas em cada corte, cada árvore fica fraca, e como o PCA concentra a informação nas primeiras componentes, o sorteio muitas vezes deixa de fora justamente a mais útil.

A principal limitação é a amostra: as árvores usam 1.500 dos 10.831 grãos de treino, e os outros modelos usam o treino inteiro, então a comparação é desfavorável a elas.

## Comparação final no teste

Todos os modelos foram escolhidos usando só o treino; aqui medimos cada um no mesmo teste de 2712 grãos. A rede neural está no notebook 05 e entra na comparação do relatório final.

In [ ]:
comparacao = [resultado_logistica, *[{k: v for k, v in r.items() if k != 'tempo_s'} for r in resultados_discriminantes], resultado_knn]
comparacao.append(metricas_classificacao('Árvore com poda (padrão, 1500 grãos)', y_previsto_poda, y_teste))
comparacao.append(metricas_classificacao('Árvore (GridSearch, 1500 grãos)', previsoes_arvores['árvore'], y_teste))
comparacao.append(metricas_classificacao(f"Bagging ({param_bagging['n_trees']} árvores, 1500 grãos)", previsoes_arvores['bagging'], y_teste))
comparacao.append(metricas_classificacao(f"Random forest ({param_r['n_trees']} árvores, 1500 grãos)", previsoes_arvores['random forest'], y_teste))

tabela_teste = pd.DataFrame(comparacao).sort_values('acuracia', ascending=False).reset_index(drop=True)
tabela_teste.round(4)

In [ ]:
salvar_resultados("classificacao", tabela_teste)

## Fechamento

No teste, o KNN (92,51%) foi o melhor classificador deste notebook, seguido de perto pela logística (91,74%), pela QDA com PCA de 99% (91,92%), pela QDA com as 16 variáveis (91,67%) e pela LDA (91,08%). As árvores ficaram perto de 89–90%, e o random forest em 75,5%. A rede neural do notebook 05 (93,1%) entra na comparação do relatório final.

Os erros se concentram nos pares que a análise exploratória já apontava, `DERMASON` × `SIRA` e `CALI` × `BARBUNYA`. Como a diferença entre KNN, logística e QDA é de menos de um ponto, a logística é uma boa escolha quando se quer explicar o modelo: tem coeficientes de leitura direta e treina rápido, enquanto o KNN precisa de padronização, PCA e escolha do k, e demora para prever.